# IEC104 Attack Classification — One-vs-Rest (OvR) Benchmark

**Dataset**: SANDI-2024 (Substation Anomaly Network Data for Intrusion detection 2024)  
**Protocol**: IEC60870-5-104 (IEC104)

## Improvements over `iec104_benchmark.ipynb`

| Issue in benchmark | Fix here |
|---|---|
| Random 70/30 split → temporal leakage | **Time-based split**: first 70% of each class's flows → train, last 30% → test |
| Single multi-class model dominated by dosattack (95%) | **Explicit One-vs-Rest**: 8 independent binary classifiers, each trained on a balanced problem |
| SMOTE synthesizes data from very small minority classes | **`class_weight='balanced_subsample'`**: re-weights samples without generating synthetic data |

## What OvR does

For each of the 8 attack classes, one binary classifier is trained:
- **Positive**: all samples of that class
- **Negative**: all samples of every other class

At inference, all 8 classifiers run in parallel. The class whose classifier returns the highest confidence score wins.  
Each binary problem is isolated — dosattack's 304k samples cannot overshadow the mitmattack classifier.

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

pd.set_option('display.max_columns', 10)
pd.set_option('display.max_rows', 20)
pd.set_option('display.float_format', '{:.4f}'.format)

# ── Paths ──────────────────────────────────────────────────────────────────
DATA_ROOT    = '/Users/metis/Library/Mobile Documents/com~apple~CloudDocs/UT-iC/01 Project/15487636'
IEC104_DIR   = os.path.join(DATA_ROOT, 'processed-iec104', 'iec104')
HEADERS_FILE = os.path.join(IEC104_DIR, 'headers_iec104.txt')
NOTEBOOKS_DIR = os.getcwd()

print('IEC104 data directory:', IEC104_DIR)
print('Notebooks dir:        ', NOTEBOOKS_DIR)
print('Data dir exists?', os.path.exists(IEC104_DIR))

: 

---
## 1. Load Data

In [ ]:
def read_headers(filename):
    with open(filename, 'r') as f:
        data = f.read()
    values = data.split(',\n')
    values[-1] = values[-1].rstrip('\n')
    return values

cols = read_headers(HEADERS_FILE)
print(f'Features: {len(cols) - 1} + 1 Label column')

In [ ]:
# Load each CSV file and tag with source filename (preserves intra-file temporal order)
csv_files = []
for root, dirs, files in os.walk(IEC104_DIR):
    for name in sorted(files):
        if name.endswith('.csv'):
            csv_files.append(os.path.join(root, name))

frames = []
for path in csv_files:
    df_tmp = pd.read_csv(path, usecols=cols)
    # Preserve row order from file — CICFlowMeter outputs flows in time order
    df_tmp['_row_order'] = range(len(df_tmp))
    df_tmp['_source_file'] = os.path.basename(path)
    frames.append(df_tmp)

df = pd.concat(frames, ignore_index=True)

# Clean NaN / Inf
df.dropna(axis=1, inplace=True)
df = df[~df.isin([np.nan, np.inf, -np.inf]).any(axis=1)]

print(f'Total rows: {len(df):,}  |  Columns: {df.shape[1]}')
print(df['Label'].value_counts())

---
## 2. Time-Based Train/Test Split

**Why this matters**: Network flows within each capture file are ordered in time. A random split leaks future flows into training (adjacent flows from the same burst end up in both sets). 

**Strategy**: For each class, sort by intra-file row order and take the first 70% as training, the last 30% as test. This simulates a real deployment scenario where the model is trained on historical data and evaluated on data that comes after.

In [ ]:
FEATURE_COLS = [c for c in df.columns if c not in ('Label', '_row_order', '_source_file')]

train_frames, test_frames = [], []

for label, group in df.groupby('Label'):
    # Sort by temporal order within the source file
    group_sorted = group.sort_values(['_source_file', '_row_order'])
    n = len(group_sorted)
    split_idx = int(n * 0.7)
    train_frames.append(group_sorted.iloc[:split_idx])
    test_frames.append(group_sorted.iloc[split_idx:])

df_train = pd.concat(train_frames).reset_index(drop=True)
df_test  = pd.concat(test_frames).reset_index(drop=True)

X_train = df_train[FEATURE_COLS].values
y_train = df_train['Label'].values
X_test  = df_test[FEATURE_COLS].values
y_test  = df_test['Label'].values

print(f'Train: {len(df_train):,} rows  |  Test: {len(df_test):,} rows')

In [ ]:
# Show split breakdown per class
split_summary = []
for label in sorted(df['Label'].unique()):
    n_train = (y_train == label).sum()
    n_test  = (y_test  == label).sum()
    split_summary.append({'Class': label, 'Train': n_train, 'Test': n_test, 'Total': n_train + n_test})

split_df = pd.DataFrame(split_summary).set_index('Class')
print('Split breakdown per class:')
print(split_df.to_string())

---
## 3. One-vs-Rest Model

We use `ExtraTreesClassifier` as the base (winner of the benchmark notebook) inside sklearn's `OneVsRestClassifier`.

**`class_weight='balanced_subsample'`**: At each tree's bootstrap sample, class weights are recomputed to be inversely proportional to class frequency. This is the appropriate variant for ensemble methods — it applies balancing locally per tree rather than globally, which is more robust. No synthetic data is generated.

In [ ]:
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

base_clf = ExtraTreesClassifier(
    n_estimators=100,
    class_weight='balanced_subsample',  # per-bootstrap reweighting — no synthetic data
    random_state=123,
    n_jobs=-1
)

ovr = OneVsRestClassifier(base_clf, n_jobs=1)  # n_jobs=1 here; parallelism is inside base_clf

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', ovr)
])

print('Fitting OvR pipeline...')
pipeline.fit(X_train, y_train)
print('Done.')

---
## 4. Evaluation

In [ ]:
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score, matthews_corrcoef, cohen_kappa_score
)

y_pred = pipeline.predict(X_test)

acc   = accuracy_score(y_test, y_pred)
f1_w  = f1_score(y_test, y_pred, average='weighted')
f1_m  = f1_score(y_test, y_pred, average='macro')
mcc   = matthews_corrcoef(y_test, y_pred)
kappa = cohen_kappa_score(y_test, y_pred)

print('=== OvR Model — Test Set Performance (time-based split) ===')
print(f'Accuracy         : {acc:.4f}')
print(f'F1 (weighted)    : {f1_w:.4f}   ← dominated by dosattack')
print(f'F1 (macro)       : {f1_m:.4f}   ← treats all classes equally')
print(f'MCC              : {mcc:.4f}')
print(f'Kappa            : {kappa:.4f}')
print()
print('Per-class report:')
print(classification_report(y_test, y_pred, zero_division=0))

In [ ]:
# Confusion matrix
labels_sorted = sorted(np.unique(y_test))
cm = confusion_matrix(y_test, y_pred, labels=labels_sorted)

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=labels_sorted, yticklabels=labels_sorted, ax=ax
)
ax.set_title('OvR (ExtraTrees) — Confusion Matrix (time-based split)')
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
plt.xticks(rotation=40, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(NOTEBOOKS_DIR, 'confusion_matrix_ovr.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Per-Binary-Classifier Analysis

One advantage of explicit OvR: we can inspect each binary classifier's confidence scores independently. This reveals how certain each sub-classifier is, which is useful for anomaly detection (low max confidence = uncertain = potential novel attack).

In [ ]:
# Confidence scores from each binary classifier
# Shape: (n_test_samples, n_classes)
X_test_scaled = pipeline.named_steps['scaler'].transform(X_test)
confidence_scores = pipeline.named_steps['clf'].predict_proba(X_test_scaled)
class_labels = pipeline.named_steps['clf'].classes_

conf_df = pd.DataFrame(confidence_scores, columns=class_labels)
conf_df['true_label'] = y_test
conf_df['pred_label'] = y_pred
conf_df['max_confidence'] = confidence_scores.max(axis=1)
conf_df['correct'] = conf_df['true_label'] == conf_df['pred_label']

print('Mean max confidence per true class:')
print(conf_df.groupby('true_label')['max_confidence'].mean().sort_values().to_string())
print()
print('Mean max confidence: correct vs incorrect predictions:')
print(conf_df.groupby('correct')['max_confidence'].describe().to_string())

In [ ]:
# Distribution of max confidence scores — split by correct/incorrect
fig, ax = plt.subplots(figsize=(9, 4))
for correct, group in conf_df.groupby('correct'):
    label = 'Correct' if correct else 'Incorrect'
    ax.hist(group['max_confidence'], bins=50, alpha=0.6, label=label)
ax.set_xlabel('Max confidence score (winning binary classifier)')
ax.set_ylabel('Count')
ax.set_title('OvR Confidence Distribution — Correct vs Incorrect Predictions')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(NOTEBOOKS_DIR, 'confidence_distribution_ovr.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Comparison: Benchmark vs OvR

**Note**: Direct metric comparison must be interpreted carefully — the benchmark used a **random split** (optimistic) while OvR uses a **time-based split** (realistic). Any drop in metrics here reflects more honest evaluation, not necessarily a worse model.

In [ ]:
comparison = pd.DataFrame({
    'Metric': ['Accuracy', 'F1 (weighted)', 'F1 (macro)', 'MCC', 'Kappa'],
    'Paper LDA (random split)': [0.8566, 0.8546, None, 0.4266, 0.4264],
    'Benchmark ExtraTrees (random split)': [1.0000, 1.0000, None, 0.9996, 0.9996],
    'OvR ExtraTrees (time-based split)': [acc, f1_w, f1_m, mcc, kappa]
}).set_index('Metric')

print(comparison.to_string())
print()
print('Key insight: if OvR (time-based) scores drop significantly vs benchmark (random),')
print('the drop is attributable to removing temporal leakage — not to OvR being worse.')

---
## 7. Save the Model

In [ ]:
import pickle
from datetime import datetime

timestamp = datetime.utcnow().strftime('%Y-%m-%d_%H-%M-%S')
model_path = os.path.join(NOTEBOOKS_DIR, f'ovr_iec104_extratrees_{timestamp}.pkl')

with open(model_path, 'wb') as f:
    pickle.dump(pipeline, f)

print(f'Model saved: {model_path}')

---
## Summary

This notebook introduced two improvements over the original benchmark:

1. **Time-based split** — removes temporal leakage; metrics now reflect real deployment conditions
2. **Explicit OvR** — isolates each attack class into its own binary decision; dosattack no longer dominates minority classifiers; no synthetic data needed

The confidence score analysis (Section 5) is the unique insight OvR enables: low max confidence on a prediction is a natural anomaly signal — it suggests the input doesn't strongly resemble any known attack class, which could indicate a novel attack type.

### Next steps (XAI)
- Apply SHAP to each of the 8 binary classifiers independently
- Compare which features each binary classifier relies on — do classifiers for similar attacks (dos, flood, ntp) share important features?
- Use confidence threshold tuning to study the false-negative vs false-positive tradeoff per class